# Setup check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/00_setup_check.ipynb)

This notebook checks that everything you need for the exercise sessions works in your Google Colab environment. It takes about two minutes.

**How to run it:** choose **Runtime → Run all** and wait until the last cell prints a summary. If everything shows ✅, you are ready. If something shows ❌, follow the advice printed next to it, or bring a screenshot of the summary to the first session.

## 1. Get the course material

This cell downloads the course repository (notebooks and data) into your Colab session. Every course notebook starts with the same cell.

In [ ]:
import os, sys

REPO_URL = "https://github.com/ivastar/ml_for_astro_notebooks.git"
REPO_DIR = "ml_for_astro_notebooks"
IN_COLAB = "google.colab" in sys.modules

if not os.path.exists(REPO_DIR) and not os.path.basename(os.getcwd()) == REPO_DIR:
    !git clone -q {REPO_URL}
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}

print("running in Colab:", IN_COLAB)
print("working directory:", os.getcwd())

## 2. Install the packages that Colab does not include

In [ ]:
%pip install -q astroquery umap-learn

## 3. Check the packages

In [ ]:
import importlib
from importlib.metadata import version, PackageNotFoundError

results = {}          # name -> (ok, message)

ICON = {True: "✅", False: "❌", "warn": "⚠️"}

def record(name, ok, msg=""):
    msg = msg if len(msg) < 200 else msg[:200] + " ..."
    results[name] = (ok, msg)
    print(f"{ICON[ok]} {name}" + (f"  —  {msg}" if msg else ""))

def vtuple(v):
    return tuple(int(x) for x in v.split(".")[:2] if x.isdigit())

required = {                      # pip name: (import name, minimum version)
    "numpy": ("numpy", "1.23"),
    "scipy": ("scipy", "1.9"),
    "pandas": ("pandas", "1.5"),
    "matplotlib": ("matplotlib", "3.6"),
    "scikit-learn": ("sklearn", "1.3"),     # 1.3 is the first version with HDBSCAN
    "astropy": ("astropy", "5.2"),
    "astroquery": ("astroquery", "0.4.6"),
    "umap-learn": ("umap", "0.5"),
}
print(f"Python {sys.version.split()[0]}\n")
for pip_name, (module, vmin) in required.items():
    try:
        importlib.import_module(module)
        v = version(pip_name)
        ok = vtuple(v) >= vtuple(vmin)
        record(f"{pip_name} {v}", ok, "" if ok else f"need >= {vmin}: run `%pip install -U {pip_name}`, then Runtime → Restart session")
    except (ImportError, PackageNotFoundError) as err:
        record(pip_name, False, f"not installed ({err}); rerun the install cell, then Runtime → Restart session")

## 4. Try out each method we will use

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

def test(name, fn, failure=False):
    try:
        msg = fn()
        record(name, True, msg or "")
    except Exception as err:
        record(name, failure, f"{type(err).__name__}: {err}")

def t_plot():
    fig, ax = plt.subplots(figsize=(5, 2.5))
    x = np.linspace(0, 4*np.pi, 200)
    ax.plot(x, np.sin(x), color="#2a78d6")
    ax.set_title("If you can see this curve, plotting works")
    plt.show()

def t_regression():
    from sklearn.linear_model import LinearRegression, Ridge
    x = np.arange(50)[:, None]; y = 3*x[:, 0] + 3 + np.random.default_rng(0).normal(0, 1, 50)
    b = LinearRegression().fit(x, y).coef_[0]
    Ridge(alpha=1).fit(x, y)
    assert abs(b - 3) < 0.1
    return f"fitted slope {b:.2f} (true 3)"

def t_classification():
    from sklearn.datasets import make_blobs
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    X, y = make_blobs(400, centers=2, random_state=0)
    Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=0)
    acc = LogisticRegression().fit(Xtr, ytr).score(Xte, yte)
    return f"test accuracy {acc:.2f}"

def t_clustering():
    from sklearn.datasets import make_blobs
    from sklearn.cluster import KMeans, HDBSCAN
    from sklearn.mixture import GaussianMixture
    X, _ = make_blobs(600, centers=[(-6, 0), (0, 6), (6, 0)], cluster_std=1.0, random_state=1)
    KMeans(3, n_init=10, random_state=0).fit(X)
    GaussianMixture(3, random_state=0).fit(X)
    n = HDBSCAN(min_cluster_size=20).fit_predict(X).max() + 1
    assert n == 3, f"HDBSCAN found {n} clusters instead of 3"
    return "k-means, GMM and HDBSCAN all ran"

def t_dimred():
    from sklearn.datasets import make_blobs
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    import umap
    X, _ = make_blobs(300, n_features=10, centers=3, random_state=2)
    PCA(2).fit_transform(X)
    TSNE(2, perplexity=20, random_state=0).fit_transform(X)
    umap.UMAP(random_state=0).fit_transform(X)
    return "PCA, t-SNE and UMAP all ran"

def t_astropy():
    import astropy.units as u
    from astropy.cosmology import Planck18
    from astropy.table import Table
    d = Planck18.comoving_distance(0.023).to(u.Mpc).value
    Table({"a": [1, 2, 3]}).write("setup_test.fits", overwrite=True)
    assert len(Table.read("setup_test.fits")) == 3
    os.remove("setup_test.fits")
    return f"comoving distance to Coma {d:.0f} Mpc"

for name, fn in [("plotting", t_plot), ("regression", t_regression), ("classification", t_classification),
                 ("clustering (k-means, GMM, HDBSCAN)", t_clustering), ("dimensionality reduction", t_dimred),
                 ("astropy (units, cosmology, FITS)", t_astropy)]:
    test(name, fn)

## 5. Check access to astronomical archives

Some notebooks download data from the SDSS and MAST archives. If this check fails, the archive may just be busy — try again later. It is not a problem with your setup.

In [ ]:
def t_sdss():
    from astroquery.sdss import SDSS
    t = SDSS.query_sql("SELECT TOP 3 ra, dec, z FROM SpecObj WHERE class = 'GALAXY'", data_release=17, timeout=60)
    assert t is not None and len(t) == 3
    return "SDSS SkyServer query returned 3 galaxies"

def t_mast():
    import urllib.request
    with urllib.request.urlopen("https://archive.stsci.edu/", timeout=30) as r:
        assert r.status == 200
    return "MAST archive reachable"

test("SDSS archive", t_sdss, failure="warn")
test("MAST archive", t_mast, failure="warn")

## 6. Summary

In [ ]:
n_bad = sum(ok is False for ok, _ in results.values())
n_warn = sum(ok == "warn" for ok, _ in results.values())
print("=" * 60)
for name, (ok, msg) in results.items():
    print(f"{ICON[ok]} {name}")
print("=" * 60)
if n_bad == 0:
    print("Your setup works — you are ready for the course. 🎉")
    if n_warn:
        print("(⚠️ An archive could not be reached right now. That is usually temporary: try again later.)")
else:
    print(f"{n_bad} check(s) failed. Scroll up for the messages next to ❌.")
    print("Most problems are fixed by Runtime → Restart session and then Runtime → Run all.")
    print("If that does not help, bring a screenshot of this summary to the first session.")